In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, month, year, round

spark = SparkSession.builder \
    .appName("Retail Store Insights") \
    .getOrCreate()

In [2]:
products_data = [
    (101, "Laptop", "Electronics", 45000, 60000),
    (102, "Mobile", "Electronics", 18000, 25000),
    (103, "Chair", "Furniture", 3000, 5000),
    (104, "Table", "Furniture", 5000, 8000)
]

products_columns = [
    "product_id",
    "product_name",
    "category",
    "cost",
    "price"
]

In [3]:
products_df = spark.createDataFrame(products_data, products_columns)

products_df.show()

+----------+------------+-----------+-----+-----+
|product_id|product_name|   category| cost|price|
+----------+------------+-----------+-----+-----+
|       101|      Laptop|Electronics|45000|60000|
|       102|      Mobile|Electronics|18000|25000|
|       103|       Chair|  Furniture| 3000| 5000|
|       104|       Table|  Furniture| 5000| 8000|
+----------+------------+-----------+-----+-----+



In [4]:
stores_data = [
    (1, "Chennai Store", "South"),
    (2, "Bangalore Store", "South"),
    (3, "Mumbai Store", "West")
]

stores_columns = [
    "store_id",
    "store_name",
    "region"
]

stores_df = spark.createDataFrame(stores_data, stores_columns)

stores_df.show()

+--------+---------------+------+
|store_id|     store_name|region|
+--------+---------------+------+
|       1|  Chennai Store| South|
|       2|Bangalore Store| South|
|       3|   Mumbai Store|  West|
+--------+---------------+------+



In [5]:
sales_data = [
    (1001, 1, 101, 5, "2026-06-01"),
    (1002, 2, 102, 8, "2026-06-01"),
    (1003, 3, 103, 10, "2026-06-02"),
    (1004, 1, 104, 6, "2026-06-03"),
    (1005, 2, 101, 7, "2026-06-04"),
    (1006, 3, 102, 12, "2026-06-04")
]

sales_columns = [
    "sale_id",
    "store_id",
    "product_id",
    "quantity",
    "sale_date"
]

sales_df = spark.createDataFrame(sales_data, sales_columns)

sales_df.show()

+-------+--------+----------+--------+----------+
|sale_id|store_id|product_id|quantity| sale_date|
+-------+--------+----------+--------+----------+
|   1001|       1|       101|       5|2026-06-01|
|   1002|       2|       102|       8|2026-06-01|
|   1003|       3|       103|      10|2026-06-02|
|   1004|       1|       104|       6|2026-06-03|
|   1005|       2|       101|       7|2026-06-04|
|   1006|       3|       102|      12|2026-06-04|
+-------+--------+----------+--------+----------+



In [7]:
from pyspark.sql.functions import to_date

sales_df=sales_df.withColumn(
    "sale_date",
    to_date(col("sale_date"),"yyyy-MM-dd")
)

sales_df.printSchema()

root
 |-- sale_id: long (nullable = true)
 |-- store_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- sale_date: date (nullable = true)



In [8]:
sales_products=sales_df.join(
    products_df,
    "product_id"
)

sales_products.show()

+----------+-------+--------+--------+----------+------------+-----------+-----+-----+
|product_id|sale_id|store_id|quantity| sale_date|product_name|   category| cost|price|
+----------+-------+--------+--------+----------+------------+-----------+-----+-----+
|       101|   1001|       1|       5|2026-06-01|      Laptop|Electronics|45000|60000|
|       101|   1005|       2|       7|2026-06-04|      Laptop|Electronics|45000|60000|
|       102|   1002|       2|       8|2026-06-01|      Mobile|Electronics|18000|25000|
|       102|   1006|       3|      12|2026-06-04|      Mobile|Electronics|18000|25000|
|       103|   1003|       3|      10|2026-06-02|       Chair|  Furniture| 3000| 5000|
|       104|   1004|       1|       6|2026-06-03|       Table|  Furniture| 5000| 8000|
+----------+-------+--------+--------+----------+------------+-----------+-----+-----+



In [9]:
sales_products=sales_products.withColumn(
    "revenue",
    col("quantity")*col("price")
)

sales_products.show()

+----------+-------+--------+--------+----------+------------+-----------+-----+-----+-------+
|product_id|sale_id|store_id|quantity| sale_date|product_name|   category| cost|price|revenue|
+----------+-------+--------+--------+----------+------------+-----------+-----+-----+-------+
|       101|   1001|       1|       5|2026-06-01|      Laptop|Electronics|45000|60000| 300000|
|       101|   1005|       2|       7|2026-06-04|      Laptop|Electronics|45000|60000| 420000|
|       102|   1002|       2|       8|2026-06-01|      Mobile|Electronics|18000|25000| 200000|
|       102|   1006|       3|      12|2026-06-04|      Mobile|Electronics|18000|25000| 300000|
|       103|   1003|       3|      10|2026-06-02|       Chair|  Furniture| 3000| 5000|  50000|
|       104|   1004|       1|       6|2026-06-03|       Table|  Furniture| 5000| 8000|  48000|
+----------+-------+--------+--------+----------+------------+-----------+-----+-----+-------+



In [11]:
underperforming_products=(
    sales_products
    .groupBy("product_id","product_name")
    .agg(
        sum("quantity").alias("total_units_sold")
    )
    .filter(col("total_units_sold")<10)
)

underperforming_products.show()

+----------+------------+----------------+
|product_id|product_name|total_units_sold|
+----------+------------+----------------+
|       104|       Table|               6|
+----------+------------+----------------+



In [12]:
monthly_revenue = (
    sales_products
    .withColumn("year", year("sale_date"))
    .withColumn("month", month("sale_date"))
    .groupBy("store_id", "year", "month")
    .agg(
        sum("revenue").alias("monthly_revenue")
    )
)

monthly_revenue.show()

+--------+----+-----+---------------+
|store_id|year|month|monthly_revenue|
+--------+----+-----+---------------+
|       3|2026|    6|         350000|
|       1|2026|    6|         348000|
|       2|2026|    6|         620000|
+--------+----+-----+---------------+



In [13]:
store_average = (
    monthly_revenue
    .join(stores_df, "store_id")
    .groupBy("store_id", "store_name")
    .agg(
        round(avg("monthly_revenue"), 2).alias("average_monthly_revenue")
    )
)

store_average.show()

+--------+---------------+-----------------------+
|store_id|     store_name|average_monthly_revenue|
+--------+---------------+-----------------------+
|       1|  Chennai Store|               348000.0|
|       3|   Mumbai Store|               350000.0|
|       2|Bangalore Store|               620000.0|
+--------+---------------+-----------------------+



In [14]:
underperforming_products.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/underperforming_products")

In [15]:
store_average.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/store_summary")

In [16]:
print("Underperforming Products")
underperforming_products.show(truncate=False)

Underperforming Products
+----------+------------+----------------+
|product_id|product_name|total_units_sold|
+----------+------------+----------------+
|104       |Table       |6               |
+----------+------------+----------------+



In [17]:
print("Store Level Summary")
store_average.show(truncate=False)

Store Level Summary
+--------+---------------+-----------------------+
|store_id|store_name     |average_monthly_revenue|
+--------+---------------+-----------------------+
|1       |Chennai Store  |348000.0               |
|3       |Mumbai Store   |350000.0               |
|2       |Bangalore Store|620000.0               |
+--------+---------------+-----------------------+

